<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_challenge_W7D2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Installation des dépendances et Chargement des données
Cette section installe `transformers`, `datasets` et `evaluate`, puis charge les tweets pour l'analyse des sentiments.

In [ ]:
!pip install -q transformers datasets evaluate accelerate

import pandas as pd
from datasets import load_dataset
import matplotlib.pyplot as plt

# 1. Chargement du dataset
dataset = load_dataset('tweet_eval', 'sentiment')

# Affichage de la répartition
print("Répartition des données :")
for split in dataset.keys():
    print(f"{split}: {len(dataset[split])} exemples")

# Conversion en DataFrame pour l'analyse
df_train = pd.DataFrame(dataset['train'])
label_map = {0: 'négatif', 1: 'neutre', 2: 'positif'}
df_train['label_name'] = df_train['label'].map(label_map)

print("\nRépartition des classes (Train) :")
display(df_train['label_name'].value_counts())

# Enregistrement d'exemples pour plus tard
examples = {}
for label_id, label_name in label_map.items():
    examples[label_name] = df_train[df_train['label'] == label_id].head(2)['text'].tolist()

print("\nExemples enregistrés pour l'inspection :")
import json
print(json.dumps(examples, indent=2))

### 2. Pipeline de Tokenisation
Initialisation du tokeniseur DistilBERT et préparation des données pour l'entraînement.

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Mapping du dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Formatage pour PyTorch
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
print("Tokenisation terminée.")